# DeepFM với dữ liệu Travela (MongoDB)

Notebook này huấn luyện DeepFM bằng **LibRecommender** trên dữ liệu thực của hệ thống Travela.

**Nguồn dữ liệu (MongoDB):**
- `tbl_reviews` → rating người dùng cho tour (label chính)
- `tbl_booking` → lượt đặt tour (positive signal)
- `tbl_user_interactions` → view/click/bookmark
- `tbl_tours` → thông tin tour (features)
- `tbl_tour_departures` → mapping departure → tour

**Features cho DeepFM:**
| Feature | Loại | Nguồn |
|---------|------|-------|
| `destination` | sparse (item) | tbl_tours.destination |
| `price_bucket` | sparse (item) | tbl_tours.priceAdult |
| `duration_bucket` | sparse (item) | tbl_tours.time |

## 1. Kết nối MongoDB & Load dữ liệu

In [21]:
import warnings
import re
import os
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# Load biến môi trường từ file .env
load_dotenv(Path("..") / ".env")

MONGO_URI = os.getenv("MONGODB_URI")
print(f"Connecting to MongoDB...")
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=10000)
db = client.get_default_database()
print(f"Database: {db.name}")

Connecting to MongoDB...
Database: travela


In [22]:
# Load tất cả collections cần thiết
print("Loading data from MongoDB...")

tours_raw = list(db.tbl_tours.find(
    {},
    {"_id": 1, "destination": 1, "priceAdult": 1, "time": 1, "images": 1, "itinerary": 1}
))

departures_raw = list(db.tbl_tour_departures.find(
    {},
    {"_id": 1, "tourId": 1}
))

reviews_raw = list(db.tbl_reviews.find(
    {},
    {"_id": 1, "userId": 1, "tourId": 1, "rating": 1}
))

bookings_raw = list(db.tbl_booking.find(
    {"bookingStatus": {"$in": ["completed", "confirmed", "pending"]}},
    {"_id": 1, "userId": 1, "tourDepartureId": 1, "bookingStatus": 1}
))

interactions_raw = list(db.tbl_user_interactions.find(
    {"type": {"$in": ["view", "click", "bookmark", "share"]}},
    {"_id": 1, "userId": 1, "tourId": 1, "type": 1}
))

print(f"  Tours:        {len(tours_raw)}")
print(f"  Departures:   {len(departures_raw)}")
print(f"  Reviews:      {len(reviews_raw)}")
print(f"  Bookings:     {len(bookings_raw)}")
print(f"  Interactions: {len(interactions_raw)}")

Loading data from MongoDB...
  Tours:        43
  Departures:   120
  Reviews:      309
  Bookings:     589
  Interactions: 1636


## 2. Tiền xử lý dữ liệu

Chuyển dữ liệu từ MongoDB thành DataFrame theo định dạng LibRecommender:
- `user`: integer ID của user
- `item`: integer ID của tour  
- `label`: điểm rating (1-5 cho rating task, hoặc 0/1 cho ranking task)
- Feature columns: destination, price_bucket, duration_bucket

In [23]:
# ── Build lookup tables ──────────────────────────────────────────────────────

# Map departure_id → tour_id
dep_to_tour = {str(d["_id"]): str(d["tourId"]) for d in departures_raw}

# Map tour_id → tour features
tour_info = {str(t["_id"]): t for t in tours_raw}

print(f"Unique destinations: {len(set(t.get('destination','') for t in tours_raw))}")
print("Sample destinations:", list(set(t.get('destination','') for t in tours_raw))[:5])

Unique destinations: 29
Sample destinations: ['Miền Trung', 'Miền Tây', 'Huế', 'Phú Quốc', 'Kiên Giang']


In [24]:
# ── Feature engineering cho tours ───────────────────────────────────────────

def parse_duration(time_str):
    """Trích xuất số ngày từ chuỗi như '3 ngày 2 đêm', '4N3D'."""
    if not time_str:
        return 1
    match = re.search(r"(\d+)\s*ng[aà]y", time_str.lower())
    if match:
        return int(match.group(1))
    match = re.search(r"(\d+)\s*n", time_str.lower())
    if match:
        return int(match.group(1))
    match = re.search(r"(\d+)", time_str)
    if match:
        return int(match.group(1))
    return 1

def get_duration_bucket(days):
    """Phân nhóm số ngày: 0=1ngày, 1=2ngày, ..., 6=7+ngày."""
    return min(max(days - 1, 0), 6)

# Tính price buckets bằng quantile từ dữ liệu thực
prices = [t.get("priceAdult", 0) for t in tours_raw if t.get("priceAdult", 0) > 0]
if prices:
    price_boundaries = [np.percentile(prices, p) for p in range(10, 100, 10)]
else:
    price_boundaries = [1e6, 2e6, 3e6, 4e6, 5e6, 6e6, 8e6, 10e6, 15e6]

def get_price_bucket(price):
    """Phân nhóm giá vào 10 bucket theo quantile."""
    for i, boundary in enumerate(price_boundaries):
        if price <= boundary:
            return i
    return 9

# Build tour features DataFrame
tour_features = []
for t in tours_raw:
    tid = str(t["_id"])
    tour_features.append({
        "tour_id_str": tid,
        "destination":     t.get("destination", "unknown") or "unknown",
        "price_bucket":    get_price_bucket(t.get("priceAdult", 0)),
        "duration_bucket": get_duration_bucket(parse_duration(t.get("time", ""))),
    })

tour_feat_df = pd.DataFrame(tour_features)
print(f"Tour features shape: {tour_feat_df.shape}")
print(f"\nPrice bucket distribution:")
print(tour_feat_df["price_bucket"].value_counts().sort_index().to_string())
print(f"\nDuration bucket distribution:")
print(tour_feat_df["duration_bucket"].value_counts().sort_index().to_string())

Tour features shape: (43, 4)

Price bucket distribution:
price_bucket
0    5
1    5
2    3
3    4
4    5
5    4
6    4
7    4
8    4
9    5

Duration bucket distribution:
duration_bucket
0     8
1     8
2    21
3     6


In [25]:
# ── Xây dựng interaction DataFrame ──────────────────────────────────────────
# Kết hợp Reviews + Bookings + Interactions thành 1 bảng thống nhất

# Trọng số label cho từng loại tín hiệu
BOOKING_LABELS = {"completed": 5.0, "confirmed": 4.0, "pending": 3.0}
INTERACTION_LABELS = {"bookmark": 4.0, "share": 3.5, "click": 2.0, "view": 1.0}

rows = []

# 1. Reviews: rating trực tiếp (nguồn tín hiệu mạnh nhất)
for r in reviews_raw:
    uid = str(r.get("userId", ""))
    tid = str(r.get("tourId", ""))
    rating = r.get("rating", 3)
    if uid and tid and tid in tour_info:
        rows.append({"user_str": uid, "tour_id_str": tid,
                     "label": float(rating), "source": "review"})

# 2. Bookings: mapping qua departure → tour
for b in bookings_raw:
    uid = str(b.get("userId", ""))
    dep_id = str(b.get("tourDepartureId", ""))
    tid = dep_to_tour.get(dep_id, "")
    status = b.get("bookingStatus", "")
    label = BOOKING_LABELS.get(status, 3.0)
    if uid and tid and tid in tour_info:
        rows.append({"user_str": uid, "tour_id_str": tid,
                     "label": label, "source": "booking"})

# 3. Interactions: view/click/bookmark/share
for i in interactions_raw:
    uid = str(i.get("userId", ""))
    tid = str(i.get("tourId", ""))
    itype = i.get("type", "view")
    label = INTERACTION_LABELS.get(itype, 1.0)
    if uid and tid and tid in tour_info:
        rows.append({"user_str": uid, "tour_id_str": tid,
                     "label": label, "source": "interaction"})

interactions_df = pd.DataFrame(rows)
print(f"Total interaction rows: {len(interactions_df)}")
print(f"Sources:\n{interactions_df['source'].value_counts().to_string()}")

Total interaction rows: 2274
Sources:
source
interaction    1616
booking         361
review          297


In [26]:
# ── Aggregate: lấy label cao nhất khi user-tour có nhiều tín hiệu ─────────────
agg_df = (
    interactions_df
    .groupby(["user_str", "tour_id_str"])["label"]
    .max()
    .reset_index()
)

print(f"Unique (user, tour) pairs: {len(agg_df)}")
print(f"Unique users:  {agg_df['user_str'].nunique()}")
print(f"Unique tours:  {agg_df['tour_id_str'].nunique()}")
print(f"\nLabel distribution:")
print(agg_df["label"].value_counts().sort_index().to_string())

Unique (user, tour) pairs: 813
Unique users:  71
Unique tours:  42

Label distribution:
label
1.0    176
2.0    132
3.0     52
3.5     25
4.0    212
5.0    216


In [27]:
# ── Encode user_id và tour_id thành số nguyên ────────────────────────────────
unique_users = agg_df["user_str"].unique()
unique_tours = agg_df["tour_id_str"].unique()

user_encoder = {uid: i for i, uid in enumerate(unique_users)}
tour_encoder = {tid: i for i, tid in enumerate(unique_tours)}

agg_df["user"] = agg_df["user_str"].map(user_encoder)
agg_df["item"] = agg_df["tour_id_str"].map(tour_encoder)

print(f"Encoded {len(user_encoder)} users, {len(tour_encoder)} tours")

Encoded 71 users, 42 tours


In [28]:
# ── Merge với tour features ───────────────────────────────────────────────────
# Encode tour_id trong tour_feat_df
tour_feat_df["item"] = tour_feat_df["tour_id_str"].map(tour_encoder)
tour_feat_df = tour_feat_df.dropna(subset=["item"])
tour_feat_df["item"] = tour_feat_df["item"].astype(int)

# Merge
data = agg_df.merge(tour_feat_df[["item", "destination", "price_bucket", "duration_bucket"]],
                    on="item", how="left")

# Điền missing values
data["destination"] = data["destination"].fillna("unknown")
data["price_bucket"] = data["price_bucket"].fillna(0).astype(int)
data["duration_bucket"] = data["duration_bucket"].fillna(0).astype(int)

# Giữ các cột cần thiết
data = data[["user", "item", "label", "destination", "price_bucket", "duration_bucket"]]
data = data.dropna()

print(f"Final dataset shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
data.head(10)

Final dataset shape: (813, 6)
Columns: ['user', 'item', 'label', 'destination', 'price_bucket', 'duration_bucket']


,user,item,label,destination,price_bucket,duration_bucket
0,0,0,2.0,Bình Thuận,4,2
1,0,1,2.0,Hà Giang,9,3
2,1,2,5.0,Kiên Giang,9,3
3,1,3,3.0,Đà Nẵng,9,3
4,1,4,3.0,Khánh Hòa,8,2
5,1,5,4.0,Thừa Thiên Huế,7,2
6,1,6,3.0,Cần Thơ,2,1
7,1,1,5.0,Hà Giang,9,3
8,1,7,4.0,Bình Thuận,1,2
9,1,8,4.0,Tây Nguyên,4,2


In [29]:
# Thống kê tổng quan
print("=== Dataset Summary ===")
print(f"Users:      {data['user'].nunique()}")
print(f"Tours:      {data['item'].nunique()}")
print(f"Pairs:      {len(data)}")
density = len(data) / (data['user'].nunique() * data['item'].nunique()) * 100
print(f"Density:    {density:.2f}%")
print(f"\nLabel stats:")
print(data['label'].describe())
print(f"\nDestination top 10:")
print(data['destination'].value_counts().head(10).to_string())

=== Dataset Summary ===
Users:      71
Tours:      42
Pairs:      813
Density:    27.26%

Label stats:
count    813.000000
mean       3.212177
std        1.522269
min        1.000000
25%        2.000000
50%        4.000000
75%        5.000000
max        5.000000
Name: label, dtype: float64

Destination top 10:
destination
Lào Cai       57
Cần Thơ       54
Miền Trung    50
Miền Tây      46
Đà Nẵng       43
Bình Thuận    42
Tây Ninh      42
Quảng Bình    40
Khánh Hòa     38
Lâm Đồng      36


## 3. Chia dữ liệu Train / Eval / Test

In [30]:
from libreco.data import random_split

# Shuffle dữ liệu
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

train_data, eval_data, test_data = random_split(
    data,
    multi_ratios=[0.8, 0.1, 0.1],
    seed=42,
)

print(f"Train: {len(train_data)} samples")
print(f"Eval:  {len(eval_data)} samples")
print(f"Test:  {len(test_data)} samples")

Train: 649 samples
Eval:  82 samples
Test:  82 samples


## 4. Xử lý Feature với DatasetFeat

- **`sparse_col`**: destination, price_bucket, duration_bucket → categorical
- **`dense_col`**: không có (tất cả features đã là categorical)
- **`item_col`**: các feature thuộc về tour

In [31]:
from libreco.data import DatasetFeat

# Định nghĩa feature columns
sparse_col = ["destination", "price_bucket", "duration_bucket"]
dense_col  = []          # không có dense feature
user_col   = []          # không có user feature (chỉ có user ID)
item_col   = ["destination", "price_bucket", "duration_bucket"]

# Build datasets
train_data, data_info = DatasetFeat.build_trainset(
    train_data,
    user_col=user_col,
    item_col=item_col,
    sparse_col=sparse_col,
    dense_col=dense_col,
)
eval_data = DatasetFeat.build_evalset(eval_data)
test_data = DatasetFeat.build_testset(test_data)

print(data_info)

n_users: 71, n_items: 42, data density: 21.7639 %


## 5. Khởi tạo & Huấn luyện DeepFM

> Dùng `task="ranking"` vì nhãn là implicit feedback (booking, view, click...)  
> Nếu chỉ dùng reviews với rating 1-5, có thể đổi sang `task="rating"`

In [32]:
from libreco.algorithms import DeepFM

model = DeepFM(
    task="ranking",
    data_info=data_info,
    embed_size=16,
    n_epochs=20,
    loss_type="cross_entropy",
    lr=1e-3,
    lr_decay=False,
    reg=None,
    batch_size=256,
    use_bn=True,
    dropout_rate=0.2,
    hidden_units=(128, 64, 32),
    multi_sparse_combiner="sqrtn",
    sampler="random",
    num_neg=1,
    seed=42,
)

print(f"Model: DeepFM | Task: {model.task}")
print(f"Users: {data_info.n_users} | Tours: {data_info.n_items}")

Model: DeepFM | Task: ranking
Users: 71 | Tours: 42


In [33]:
model.fit(
    train_data,
    neg_sampling=True,
    verbose=2,
    shuffle=True,
    eval_data=eval_data,
    metrics=["loss", "roc_auc", "precision", "recall", "ndcg"],
)

Training start time: 2026-05-25 12:59:26


ValueError: Variable linear_user_feat already exists, disallowed. Did you mean to set reuse=True or reuse=tf.AUTO_REUSE in VarScope? Originally defined at:

  File "/Users/qhung/KLTN/recommendation-service/venv/lib/python3.9/site-packages/libreco/algorithms/deepfm.py", line 186, in _build_user_item
  File "/Users/qhung/KLTN/recommendation-service/venv/lib/python3.9/site-packages/libreco/algorithms/deepfm.py", line 154, in build_model
  File "/Users/qhung/KLTN/recommendation-service/venv/lib/python3.9/site-packages/libreco/bases/tf_base.py", line 124, in fit
  File "/var/folders/h8/lcvjdnkj43s36b0y0p3m7hpm0000gn/T/ipykernel_44659/2937686722.py", line 1, in <module>
  File "/Users/qhung/KLTN/recommendation-service/venv/lib/python3.9/site-packages/IPython/core/interactiveshell.py", line 3550, in run_code


## 6. Đánh giá trên Test Set

In [ ]:
from libreco.evaluation import evaluate

test_result = evaluate(
    model=model,
    data=test_data,
    neg_sampling=True,
    metrics=["loss", "roc_auc", "precision", "recall", "ndcg"],
)

print("=== Kết quả Test Set ===")
for k, v in test_result.items():
    print(f"  {k:15s}: {v:.4f}")

random neg item sampling elapsed: 0.000s


eval_listwise: 100%|██████████| 1/1 [00:00<00:00, 407.29it/s]

=== Kết quả Test Set ===
  loss           : 0.6886
  roc_auc        : 0.5686
  precision      : 0.0723
  recall         : 0.3972
  ndcg           : 0.3006


## 7. Thử nghiệm Recommend & Predict

In [ ]:
# Lấy một user thực từ dữ liệu để test
sample_user_str = agg_df["user_str"].iloc[0]
sample_user_id  = user_encoder[sample_user_str]

print(f"User MongoDB ID: {sample_user_str}")
print(f"User encoded ID: {sample_user_id}")

# Tours mà user đã tương tác
user_history = agg_df[agg_df["user_str"] == sample_user_str]
print(f"\nLịch sử tương tác của user ({len(user_history)} tours):")
for _, row in user_history.iterrows():
    tid = row["tour_id_str"]
    dest = tour_info.get(tid, {}).get("destination", "?")
    print(f"  Tour {tid[:8]}... | Dest: {dest:20s} | Label: {row['label']}")

User MongoDB ID: 675abdf8fa15a7ce8be44a65
User encoded ID: 0

Lịch sử tương tác của user (2 tours):
  Tour 695a162f... | Dest: Bình Thuận           | Label: 2.0
  Tour 695a162f... | Dest: Hà Giang             | Label: 2.0


In [ ]:
# Đề xuất top-10 tours cho user
recommendations = model.recommend_user(user=sample_user_id, n_rec=10)

# Decode tour ID ngược lại sang MongoDB ID
tour_decoder = {v: k for k, v in tour_encoder.items()}

print(f"Top-10 tours được đề xuất cho user {sample_user_str[:8]}...:")
print(f"{'Rank':>4} | {'Tour ID':>10} | {'Destination':20} | {'Price Bucket':>12} | {'Duration Bucket':>15}")
print("-" * 75)

for rank, item_id in enumerate(recommendations[sample_user_id], 1):
    tour_id_str = tour_decoder.get(item_id, "?")
    tour = tour_info.get(tour_id_str, {})
    dest = tour.get("destination", "?")
    price = tour.get("priceAdult", 0)
    pb = get_price_bucket(price)
    time_str = tour.get("time", "?")
    days = parse_duration(time_str)
    db_bucket = get_duration_bucket(days)
    print(f"  {rank:2d} | {tour_id_str[:10]:>10} | {dest:20} | {pb:>12} | {db_bucket:>15}")

Top-10 tours được đề xuất cho user 675abdf8...:
Rank |    Tour ID | Destination          | Price Bucket | Duration Bucket
---------------------------------------------------------------------------
   1 | 69f7288c40 | Lào Cai              |            5 |               2
   2 | 69f7233d40 | Phú Yên              |            2 |               1
   3 | 695a162f47 | Cần Thơ              |            2 |               1
   4 | 69f7288c40 | Đà Nẵng              |            5 |               2
   5 | 69f71f9a40 | Tây Nguyên           |            4 |               2
   6 | 69f7233d40 | Vũng Tàu             |            1 |               1
   7 | 695a162f47 | Quảng Ninh           |            4 |               2
   8 | 69f7186385 | Nghệ An              |            0 |               0
   9 | 69f721c140 | Đông Bắc             |            6 |               2
  10 | 69e9eff25b | Bình Thuận           |            1 |               2


In [ ]:
# Dự đoán score cho 1 cặp (user, tour)
sample_tour_str = agg_df["tour_id_str"].iloc[0]
sample_tour_id  = tour_encoder[sample_tour_str]

score = model.predict(user=sample_user_id, item=sample_tour_id)
print(f"Dự đoán score: user={sample_user_id}, tour={sample_tour_id} → {score:.4f}")

Dự đoán score: user=0, tour=0 → 0.5055


## 8. Lưu mô hình

In [ ]:
import pickle
from libreco.data import DataInfo

save_path = "../models/deepfm_libreco_travela"
os.makedirs(save_path, exist_ok=True)

# Lưu LibRecommender model
data_info.save(save_path, model_name="deepfm")
model.save(save_path, model_name="deepfm")

# Lưu encoder mappings (để dùng trong production)
encoders = {
    "user_encoder":    user_encoder,
    "tour_encoder":    tour_encoder,
    "tour_decoder":    tour_decoder,
    "price_boundaries": price_boundaries,
}
with open(f"{save_path}/encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)

print(f"Đã lưu vào: {save_path}")
print("Files:", os.listdir(save_path))

Đã lưu vào: ../models/deepfm_libreco_travela
Files: ['deepfm_user_consumed.pkl', 'deepfm_data_info_name_mapping.json', 'encoders.pkl', 'deepfm_data_info.npz', 'deepfm_default_recs.npz', 'deepfm_item_consumed.pkl', 'deepfm_tf_variables.npz', 'deepfm_hyper_parameters.json']


## 9. Tải lại và kiểm tra

In [ ]:
# Load lại model
loaded_data_info = DataInfo.load(save_path, model_name="deepfm")
loaded_model = DeepFM.load(
    path=save_path,
    model_name="deepfm",
    data_info=loaded_data_info,
)

# Load encoders
with open(f"{save_path}/encoders.pkl", "rb") as f:
    loaded_encoders = pickle.load(f)

# Kiểm tra recommend sau khi load
recs_after_load = loaded_model.recommend_user(user=sample_user_id, n_rec=5)
print("Model đã load thành công!")
print(f"Recommended tours: {recs_after_load[sample_user_id]}")

ValueError: Variable linear_user_feat already exists, disallowed. Did you mean to set reuse=True or reuse=tf.AUTO_REUSE in VarScope? Originally defined at:

  File "/Users/qhung/KLTN/recommendation-service/venv/lib/python3.9/site-packages/libreco/algorithms/deepfm.py", line 186, in _build_user_item
  File "/Users/qhung/KLTN/recommendation-service/venv/lib/python3.9/site-packages/libreco/algorithms/deepfm.py", line 154, in build_model
  File "/Users/qhung/KLTN/recommendation-service/venv/lib/python3.9/site-packages/libreco/bases/tf_base.py", line 124, in fit
  File "/var/folders/h8/lcvjdnkj43s36b0y0p3m7hpm0000gn/T/ipykernel_44659/2937686722.py", line 1, in <module>
  File "/Users/qhung/KLTN/recommendation-service/venv/lib/python3.9/site-packages/IPython/core/interactiveshell.py", line 3550, in run_code


## 10. Retrain với dữ liệu mới (Online Update)

Theo hướng dẫn LibRecommender: khi có dữ liệu tương tác mới, ta dùng `DatasetFeat.merge_trainset` để gộp với data cũ, sau đó dùng `rebuild_model` để kế thừa trọng số đã học — **không cần train lại từ đầu**.

## 11. Thử nghiệm với task="rating" (optional)

Nếu chỉ muốn dùng **reviews** (có rating 1-5 rõ ràng), có thể train với `task="rating"` để dự đoán điểm số thay vì ranking.

## 10. Thử nghiệm với task="rating" (optional)

Nếu chỉ muốn dùng **reviews** (có rating 1-5 rõ ràng), có thể train với `task="rating"` để dự đoán điểm số thay vì ranking.

In [ ]:
# Chỉ dùng reviews thuần túy cho rating task
reviews_only = []
for r in reviews_raw:
    uid = str(r.get("userId", ""))
    tid = str(r.get("tourId", ""))
    rating = r.get("rating", 3)
    if uid and tid and tid in tour_info:
        reviews_only.append({"user_str": uid, "tour_id_str": tid, "label": float(rating)})

reviews_df = pd.DataFrame(reviews_only)
print(f"Reviews thuần túy: {len(reviews_df)} records")
print(f"Users: {reviews_df['user_str'].nunique()}, Tours: {reviews_df['tour_id_str'].nunique()}")
print(f"\nRating distribution:")
print(reviews_df['label'].value_counts().sort_index().to_string())

## Tóm tắt Pipeline (theo hướng dẫn LibRecommender)

| Bước | Mô tả | API LibRecommender |
|------|-------|-------------------|
| 1 | Cài đặt thư viện | `pip install LibRecommender` |
| 2 | Load & tiền xử lý dữ liệu | MongoDB → pandas DataFrame |
| 3 | Chia dữ liệu | `random_split(data, multi_ratios=[0.8,0.1,0.1])` |
| 4 | Xử lý features | `DatasetFeat.build_trainset(sparse_col, item_col)` |
| 5 | Huấn luyện DeepFM | `DeepFM.fit(neg_sampling=True)` |
| 6 | Đánh giá | `evaluate(metrics=[roc_auc, precision, recall, ndcg])` |
| 7 | Recommend & Predict | `recommend_user(n_rec=10)`, `predict(user, item)` |
| 8 | Lưu mô hình | `data_info.save()`, `model.save()` |
| 9 | Tải lại mô hình | `DataInfo.load()`, `DeepFM.load()` |
| 10 | Retrain dữ liệu mới | `DatasetFeat.merge_trainset()`, `rebuild_model()` |

```
MongoDB (tbl_reviews, tbl_booking, tbl_user_interactions, tbl_tours)
    ↓ (merge + aggregate max label per user-tour)
pandas DataFrame [user, item, label, destination, price_bucket, duration_bucket]
    ↓ random_split (80/10/10)
DatasetFeat.build_trainset()
    ↓
DeepFM(task="ranking", embed_size=16, hidden_units=(128,64,32))
    ↓ fit(neg_sampling=True)
evaluate() → roc_auc, precision, recall, ndcg
    ↓
recommend_user() → top-N tour IDs
```

**Nguồn dữ liệu & Label:**
| Nguồn | Label |
|-------|-------|
| tbl_reviews | rating 1–5 (explicit) |
| tbl_booking completed | 5.0 |
| tbl_booking confirmed | 4.0 |
| tbl_booking pending | 3.0 |
| tbl_user_interactions bookmark | 4.0 |
| tbl_user_interactions share | 3.5 |
| tbl_user_interactions click | 2.0 |
| tbl_user_interactions view | 1.0 |

## Tóm tắt Pipeline

```
MongoDB
├── tbl_reviews      → label = rating (1-5)
├── tbl_booking      → label = {completed:5, confirmed:4, pending:3}
├── tbl_user_interactions → label = {bookmark:4, share:3.5, click:2, view:1}
└── tbl_tours        → features: destination, price_bucket, duration_bucket
         ↓
    Merge & Aggregate (max label per user-tour pair)
         ↓
    DatasetFeat.build_trainset()
         ↓
    DeepFM.fit()  →  evaluate()  →  recommend_user()
```

| Metric | Ý nghĩa |
|--------|---------|
| `roc_auc` | Khả năng phân biệt tour tốt/xấu |
| `precision@k` | Tỉ lệ tours relevant trong top-k |
| `recall@k` | Tỉ lệ tours relevant được tìm thấy |
| `ndcg@k` | Chất lượng xếp hạng (có trọng số vị trí) |